In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dhg

from dhg.data import CoauthorshipCora
from dhg.nn import HGNNConv
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# === 1. 加载数据 ===
data = CoauthorshipCora()
num_v = data['num_vertices']
X, y = data['features'], data['labels']
train_mask, test_mask = data['train_mask'], data['test_mask']

# === 2. 手动实现特征归一化 (替代 dhg.transforms) ===
# 原理：将每一行（每个节点）的特征向量除以其总和，使特征分布在 [0, 1] 之间
row_sum = torch.sum(X, dim=1, keepdim=True)
row_sum[row_sum == 0] = 1.0  # 防止除以 0
X = X / row_sum

# === 3. 手动构建带自环的超图 ===
full_edge_list = data['edge_list'] + [(i,) for i in range(num_v)]
hg = dhg.Hypergraph(num_v, full_edge_list)

# 3. 定义 HGNN 模型
class FinalOptimizedHGNN(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.conv1 = HGNNConv(in_ch, hid_ch)
        self.conv2 = HGNNConv(hid_ch, out_ch)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, hg):
        # 第一层卷积：节点 -> 超边 -> 节点
        x = F.relu(self.conv1(x, hg))
        x = self.dropout(x)
        # 第二层卷积：提取最终分类特征
        x_out = self.conv2(x, hg)
        return x_out, x

# 4. 训练配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 将隐藏层调至 128 维，增强高维特征表达能力
model = FinalOptimizedHGNN(X.shape[1], 128, data['num_classes']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

X, y, hg = X.to(device), y.to(device), hg.to(device)

# 5. 训练循环
print(f"{'Epoch':<8} | {'Loss':<10} | {'Test Acc':<10}")
print("-" * 35)

best_acc = 0
best_features = None

for epoch in range(201):
    model.train()
    optimizer.zero_grad()
    out, features = model(X, hg)
    loss = F.cross_entropy(out[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            acc = (pred[test_mask] == y[test_mask]).float().mean()
            print(f"{epoch:<8} | {loss.item():<10.4f} | {acc:<10.2%}")
            if acc > best_acc: 
                best_acc = acc
                best_features = features.detach() # 核心优化 C: 彻底解耦梯度

print("-" * 35)
print(f"🎉 最终优化完成！最高准确率: {best_acc:.2%}")

# 6. t-SNE 聚类可视化
print("正在生成 t-SNE 可视化图...")
tsne = TSNE(n_components=2, random_state=42).fit_transform(best_features.cpu().numpy())

plt.figure(figsize=(10, 7))
scatter = plt.scatter(tsne[:, 0], tsne[:, 1], c=y.cpu().numpy(), cmap='rainbow', s=10, alpha=0.7)
plt.colorbar(scatter)
plt.title(f"Final HGNN on Native CoauthorshipCora\nBest Test Accuracy: {best_acc:.2%}")
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

Epoch    | Loss       | Test Acc  
-----------------------------------
0        | 1.9495     | 14.68%    
20       | 1.9065     | 19.70%    
40       | 1.8354     | 22.00%    
60       | 1.6731     | 25.66%    
80       | 1.6117     | 27.06%    
100      | 1.6139     | 25.97%    
120      | 1.6459     | 26.87%    
140      | 1.5704     | 26.52%    
160      | 1.6462     | 26.29%    
180      | 1.6665     | 26.40%    
200      | 1.5803     | 26.71%    
-----------------------------------
🎉 最终优化完成！最高准确率: 27.06%
正在生成 t-SNE 可视化图...


KeyboardInterrupt: 